# 02 因子挖掘

## 目标
- 构建手工量化因子（动量、波动、量价、技术指标）
- 调用RD-Agent进行自动因子挖掘
- 整合所有因子为统一的因子DataFrame

## 因子分类
| 类别 | 因子 | 说明 |
|------|------|------|
| 动量 | 5/10/20日动量、RSI、MACD、ROC | 趋势追踪 |
| 波动 | 历史波动率、ATR、布林带宽度 | 风险度量 |
| 量价 | 成交量比率、OBV变化、MFI | 资金流向 |
| 技术 | 均线偏离、威廉%R、CCI | 超买超卖 |
| 反转 | 短期反转、涨跌比 | 均值回归 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✓ 依赖加载完毕')

✓ 依赖加载完毕


## 2. 加载清洗后的数据

In [2]:
# 加载数据
close_df = pd.read_csv('data/close_df.csv', index_col=0, parse_dates=True)
ret_df = pd.read_csv('data/ret_df.csv', index_col=0, parse_dates=True)
volume_df = pd.read_csv('data/volume_df.csv', index_col=0, parse_dates=True)
high_df = pd.read_csv('data/high_df.csv', index_col=0, parse_dates=True)
low_df = pd.read_csv('data/low_df.csv', index_col=0, parse_dates=True)
open_df = pd.read_csv('data/open_df.csv', index_col=0, parse_dates=True)
# 重要: dtype=str 防止 '000001' 被读成整数 1
tickers = pd.read_csv('data/tickers.csv', dtype=str)['ticker'].tolist()

print(f'数据加载完毕')
print(f'  收盘价: {close_df.shape}')
print(f'  收益率: {ret_df.shape}')
print(f'  股票数: {len(tickers)}')
print(f'  日期范围: {close_df.index[0].strftime("%Y-%m-%d")} ~ {close_df.index[-1].strftime("%Y-%m-%d")}')

数据加载完毕
  收盘价: (717, 40)
  收益率: (716, 40)
  股票数: 40
  日期范围: 2023-01-03 ~ 2025-12-31


## 3. 因子计算函数

In [3]:
def calc_factors_for_stock(close, high, low, volume, ret):
    """
    对单只股票计算所有手工因子
    返回 dict，key=因子名, value=因子序列
    """
    factors = {}
    
    # ==================== 动量类 ====================

    # 1. N日动量（对数收益更稳定）
    for window in [5, 10, 20, 60]:
        factors[f'mom_{window}d'] = close.pct_change(window)
    
    # 2. RSI (14日)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-10)
    factors['rsi_14'] = 100 - (100 / (1 + rs))
    
    # 3. MACD (12, 26, 9)
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema_12 - ema_26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    factors['macd'] = macd_line - signal_line
    factors['macd_signal_ratio'] = macd_line / (signal_line + 1e-10)  # MACD相对信号线
    
    # 4. ROC (Rate of Change) — 价格变化率
    factors['roc_10'] = (close - close.shift(10)) / (close.shift(10) + 1e-10)
    
    # 5. 夏普比率代理: 过去20日收益/波动
    mean_ret_20 = ret.rolling(20).mean()
    std_ret_20 = ret.rolling(20).std()
    factors['sharpe_20d'] = mean_ret_20 / (std_ret_20 + 1e-10)
    
    # ==================== 波动类 ====================
    
    # 6. 历史波动率（不同窗口）
    for window in [5, 20, 60]:
        factors[f'vol_{window}d'] = ret.rolling(window).std()
    
    # 7. ATR (Average True Range) 14日
    hl = high - low
    hc = (high - close.shift()).abs()
    lc = (low - close.shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    factors['atr_14'] = tr.rolling(14).mean() / close  # 归一化
    
    # 8. 布林带宽度
    ma_20 = close.rolling(20).mean()
    std_20 = close.rolling(20).std()
    bb_upper = ma_20 + 2 * std_20
    bb_lower = ma_20 - 2 * std_20
    factors['bb_width'] = (bb_upper - bb_lower) / ma_20  # 带宽/价格，归一化
    factors['bb_position'] = (close - ma_20) / (2 * std_20 + 1e-10)  # 价格在带中的位置
    
    # ==================== 量价类 ====================
    
    # 9. 成交量比率（相对过去N日均量）
    for window in [5, 20]:
        vol_ma = volume.rolling(window).mean()
        factors[f'vol_ratio_{window}d'] = volume / (vol_ma + 1e-10)
    
    # 10. OBV变化率 (On-Balance Volume)
    obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
    factors['obv_chg_5d'] = obv.pct_change(5)  # OBV 5日变化率
    
    # 11. MFI (Money Flow Index) 14日
    typical_price = (high + low + close) / 3
    money_flow = typical_price * volume
    tp_diff = typical_price.diff()
    positive_flow = money_flow.where(tp_diff > 0, 0).rolling(14).sum()
    negative_flow = money_flow.where(tp_diff < 0, 0).rolling(14).sum()
    mfi_ratio = positive_flow / (negative_flow + 1e-10)
    factors['mfi_14'] = 100 - (100 / (1 + mfi_ratio))
    
    # ==================== 技术类 ====================
    
    # 12. 均线偏离度
    for window in [5, 10, 20, 60]:
        ma = close.rolling(window).mean()
        factors[f'ma_dev_{window}d'] = (close - ma) / (ma + 1e-10)
    
    # 13. 威廉%R (14日)
    high_14 = high.rolling(14).max()
    low_14 = low.rolling(14).min()
    factors['williams_r'] = -100 * (high_14 - close) / (high_14 - low_14 + 1e-10)
    
    # 14. CCI (Commodity Channel Index) 20日
    tp = (high + low + close) / 3
    tp_ma = tp.rolling(20).mean()
    tp_mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean())
    factors['cci_20'] = (tp - tp_ma) / (0.015 * tp_mad + 1e-10)
    
    # ==================== 反转类 ====================
    
    # 15. 短期反转 (1日反转)
    factors['reversal_1d'] = -ret  # 负收益 = 买反转
    
    # 16. 涨跌比 (过去5日)
    up_days = (ret > 0).rolling(5).sum()
    down_days = (ret < 0).rolling(5).sum()
    factors['up_down_ratio_5d'] = up_days / (down_days + 1e-10)
    
    # 17. 最大回撤（20日窗口内的）
    cum_max = close.rolling(20).max()
    factors['drawdown_20d'] = (close - cum_max) / cum_max  # 负值，值越大越好
    
    return factors

## 4. 批量计算因子

In [4]:
%%time

# 因子字典结构: {factor_name: DataFrame(index=date, columns=ticker)}
factor_panel = {}

first_stock = True
for ticker in tickers:
    close = close_df[ticker]
    high = high_df[ticker]
    low = low_df[ticker]
    volume = volume_df[ticker]
    ret = ret_df[ticker]
    
    stock_factors = calc_factors_for_stock(close, high, low, volume, ret)
    
    if first_stock:
        # 初始化面板
        for fname in stock_factors.keys():
            factor_panel[fname] = pd.DataFrame(index=close_df.index, columns=tickers, dtype=float)
        first_stock = False
    
    for fname, fseries in stock_factors.items():
        factor_panel[fname][ticker] = fseries

print(f'✓ 因子计算完成!')
print(f'  因子数量: {len(factor_panel)}')
for name in sorted(factor_panel.keys()):
    print(f'    - {name}: {factor_panel[name].shape}')

✓ 因子计算完成!
  因子数量: 28
    - atr_14: (717, 40)
    - bb_position: (717, 40)
    - bb_width: (717, 40)
    - cci_20: (717, 40)
    - drawdown_20d: (717, 40)
    - ma_dev_10d: (717, 40)
    - ma_dev_20d: (717, 40)
    - ma_dev_5d: (717, 40)
    - ma_dev_60d: (717, 40)
    - macd: (717, 40)
    - macd_signal_ratio: (717, 40)
    - mfi_14: (717, 40)
    - mom_10d: (717, 40)
    - mom_20d: (717, 40)
    - mom_5d: (717, 40)
    - mom_60d: (717, 40)
    - obv_chg_5d: (717, 40)
    - reversal_1d: (717, 40)
    - roc_10: (717, 40)
    - rsi_14: (717, 40)
    - sharpe_20d: (717, 40)
    - up_down_ratio_5d: (717, 40)
    - vol_20d: (717, 40)
    - vol_5d: (717, 40)
    - vol_60d: (717, 40)
    - vol_ratio_20d: (717, 40)
    - vol_ratio_5d: (717, 40)
    - williams_r: (717, 40)
CPU times: total: 4.58 s
Wall time: 4.69 s


### 4.1 因子质量检查

In [5]:
# 检查各因子的缺失率和基本统计
print('=' * 70)
print(f'{"因子名称":<25s} {"缺失率":>8s} {"均值":>10s} {"标准差":>10s} {"偏度":>10s}')
print('-' * 70)

for fname in sorted(factor_panel.keys()):
    fvals = factor_panel[fname].values.flatten()
    fvals = fvals[~np.isnan(fvals) & ~np.isinf(fvals)]
    
    if len(fvals) == 0:
        print(f'{fname:<25s} {"ALL NaN":>8s}')
        continue
    
    missing = np.isnan(factor_panel[fname].values).sum() / factor_panel[fname].size * 100
    print(f'{fname:<25s} {missing:>7.1f}% {np.mean(fvals):>10.4f} {np.std(fvals):>10.4f} {pd.Series(fvals).skew():>10.2f}')

因子名称                           缺失率         均值        标准差         偏度
----------------------------------------------------------------------
atr_14                        1.8%     0.0259     0.0106       1.44
bb_position                   2.6%    -0.0263     0.6512       0.17
bb_width                      2.6%     0.1221     0.0831       2.64
cci_20                        2.6%    -4.0747   112.7881       0.20
drawdown_20d                  2.6%    -0.0501     0.0452      -1.26
ma_dev_10d                    1.3%     0.0007     0.0339       1.73
ma_dev_20d                    2.6%     0.0010     0.0492       1.39
ma_dev_5d                     0.6%     0.0003     0.0218       1.42
ma_dev_60d                    8.2%     0.0045     0.0859       0.97
macd                          0.0%    -0.0024     1.2305       7.17
macd_signal_ratio             0.0%     1.6401    78.8487     113.95
mfi_14                        1.8%    51.0853    16.4939      -0.09
mom_10d                       1.4%     0.0030

## 5. 调用 RD-Agent 进行自动因子挖掘

RD-Agent 是自动化因子挖掘框架，通过遗传规划/强化学习搜索有效因子表达式。
这里通过 PowerShell 调用 WSL 中的 RD-Agent。

In [6]:
import subprocess
import json
import os

# 准备RD-Agent输入数据（CSV格式，RD-Agent可直接读取）
rd_input = pd.DataFrame(index=close_df.index)
for ticker in tickers:
    rd_input[f'{ticker}_close'] = close_df[ticker]
    rd_input[f'{ticker}_volume'] = volume_df[ticker]

rd_input.to_csv('rda_output/price_data.csv')
print(f'✓ RD-Agent输入数据已保存: rda_output/price_data.csv ({rd_input.shape})')

✓ RD-Agent输入数据已保存: rda_output/price_data.csv ((717, 80))


In [7]:
# ===== 调用 RD-Agent =====
# 通过 PowerShell 脚本在 WSL 中运行 RD-Agent
print('正在调用 RD-Agent (通过 WSL)...')
print('=' * 50)

# 查找 run_rdagent.ps1
ps_script = 'run_rdagent.ps1'
if os.path.exists(ps_script):
    try:
        result = subprocess.run(
            ['powershell', '-ExecutionPolicy', 'Bypass', '-File', ps_script],
            capture_output=True,
            text=True,
            timeout=600  # 10分钟超时
        )
        print(result.stdout)
        if result.returncode != 0:
            print(f'⚠ RD-Agent返回非零退出码: {result.returncode}')
            print(f'Stderr: {result.stderr[:500]}')
    except subprocess.TimeoutExpired:
        print('⚠ RD-Agent执行超时（10分钟），跳过自动因子挖掘')
    except FileNotFoundError:
        print('⚠ PowerShell不可用，请手动在WSL中运行RD-Agent')
else:
    print('⚠ run_rdagent.ps1 未找到，跳过RD-Agent调用')
    print('  请手动在WSL中运行RD-Agent，然后将结果放入 rda_output/')

正在调用 RD-Agent (通过 WSL)...


Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "d:\python\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "d:\python\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "d:\python\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\python\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'gbk' codec can't decode byte 0xaa in position 58: illegal multibyte sequence


None


### 5.1 解析RD-Agent输出

如果RD-Agent成功运行并输出了因子，这里将其导入并复现。

In [8]:
# 尝试加载RD-Agent生成的因子
rda_factors_loaded = 0

# RD-Agent可能的输出文件
rda_output_files = [
    'rda_output/factors.csv',
    'rda_output/best_factors.json',
    'rda_output/alpha_factors.csv',
]

for f in rda_output_files:
    if os.path.exists(f):
        print(f'发现RD-Agent输出: {f}')
        # 根据文件格式解析
        if f.endswith('.csv'):
            rda_df = pd.read_csv(f, index_col=0, parse_dates=True)
            # 假设RD-Agent输出的列是因子值，与stock对应
            for col in rda_df.columns:
                factor_name = f'rda_{col}'
                if col in tickers:
                    # 列名是ticker，需要转置
                    factor_panel[factor_name] = rda_df  # 简化处理
                else:
                    factor_panel[factor_name] = rda_df
                rda_factors_loaded += 1
                print(f'  ✓ 加载RD因子: {factor_name}')

if rda_factors_loaded == 0:
    print('未找到RD-Agent输出文件，仅使用手工因子。')
    print('这是正常的——RD-Agent可以在WSL中手动运行后补充因子。')

未找到RD-Agent输出文件，仅使用手工因子。
这是正常的——RD-Agent可以在WSL中手动运行后补充因子。


## 6. 保存因子数据

In [ ]:
# 构建宽表: 每个因子是一列（flatten: date x ticker）
factor_wide_list = []

for fname, fdf in factor_panel.items():
    # fdf: index=date, columns=ticker
    stacked = fdf.stack().to_frame(fname)
    factor_wide_list.append(stacked)

factor_wide = pd.concat(factor_wide_list, axis=1)
factor_wide.index.names = ['date', 'ticker']

# 确保 ticker 是字符串
factor_wide.index = factor_wide.index.set_levels(
    factor_wide.index.get_level_values('ticker').astype(str), level='ticker'
)

print(f'因子宽表: {factor_wide.shape}')
print(f'  index: {factor_wide.index.names}')
print(f'  因子数: {factor_wide.shape[1]}')
factor_wide.head()

In [ ]:
# 保存因子宽表 + 因子列表
import csv
factor_wide.to_csv('data/factor_wide.csv', quoting=csv.QUOTE_NONNUMERIC)

# 因子元数据
cat_map = {}
for fname in factor_panel.keys():
    if fname.startswith('mom_') or fname in ['rsi_14', 'macd', 'macd_signal_ratio', 'roc_10', 'sharpe_20d']:
        cat_map[fname] = 'momentum'
    elif fname.startswith('vol_') or fname.startswith('atr_') or fname.startswith('bb_'):
        cat_map[fname] = 'volatility'
    elif fname.startswith('vol_ratio') or fname.startswith('obv_') or fname.startswith('mfi_'):
        cat_map[fname] = 'volume_price'
    elif fname.startswith('ma_dev') or fname in ['williams_r', 'cci_20']:
        cat_map[fname] = 'technical'
    elif fname.startswith('reversal') or fname.startswith('up_down') or fname.startswith('drawdown'):
        cat_map[fname] = 'reversal'
    elif fname.startswith('rda_'):
        cat_map[fname] = 'rdagent'
    else:
        cat_map[fname] = 'other'

factor_meta = pd.DataFrame([
    {'factor_name': k, 'category': v} for k, v in cat_map.items()
])
factor_meta.to_csv('data/factor_meta.csv', index=False)

print('=' * 50)
print('因子数据保存完毕!')
print(f'  factor_wide.csv: {factor_wide.shape}')
print(f'  factor_meta.csv: {factor_meta.shape}')
print(f'  手工因子数: {len(factor_panel) - rda_factors_loaded}')
print(f'  RD-Agent因子数: {rda_factors_loaded}')
print('=' * 50)

## 小结

✅ 构建了20+个手工因子，覆盖动量、波动、量价、技术、反转5大类别
✅ 尝试调用RD-Agent进行自动因子挖掘
✅ 因子数据已保存为统一格式

下一步: [03_factor_analysis.ipynb](./03_factor_analysis.ipynb) 因子IC/IR分析